# MuJoCo 机械臂项目说明（MTD3 笔记风格）

本项目是纯 MuJoCo 的机械臂强化学习工程，支持 `SAC / PPO / TD3`，支持 `UR5_CXY` 与 `zero+Robotiq` 两套模型，支持中断保存与恢复训练。

## 1. 项目目标

训练机械臂完成到点任务：

- 环境：Gymnasium + MuJoCo
- 算法：Stable-Baselines3（SAC / PPO / TD3）
- 判定：两指中心点到目标点

## 2. 快速开始

### 2.1 安装依赖


In [ ]:
cd /home/zzyfan/mujoco_ur5_rl
pip install -r requirements.txt


### 2.2 冒烟测试（随机策略）


In [ ]:
cd /home/zzyfan/mujoco_ur5_rl
python classic/test.py --random-policy --episodes 1 --render-mode human


### 2.3 开始训练（推荐先无渲染）


In [ ]:
cd /home/zzyfan/mujoco_ur5_rl
python classic/train.py --algo sac --robot ur5_cxy --run-name exp_sac --timesteps 500000 --device cuda --no-render


## 3. 常用训练命令

### 3.1 训练时显示 MuJoCo 界面


In [ ]:
python classic/train.py --algo sac --robot ur5_cxy --run-name exp_sac_vis --timesteps 500000 --device cuda --render --render-mode human --n-envs 1


### 3.2 使用 zero 机械臂 + Robotiq 夹爪


In [ ]:
python classic/train.py --algo sac --robot zero_robotiq --run-name exp_zero --timesteps 500000 --device cuda --no-render


### 3.3 关闭评估（训练结束更快）


In [ ]:
python classic/train.py --algo sac --run-name exp_fast_exit --timesteps 500000 --eval-freq 0 --no-render


## 4. 模型保存结构（按后端+算法+机械臂独立）


In [ ]:
models/classic/{algo}/{robot}/{run_name}/
  final/
    model.zip
    vec_normalize.pkl
    replay_buffer.pkl        # SAC/TD3
  interrupted/
    model.zip
    vec_normalize.pkl
    replay_buffer.pkl        # SAC/TD3


logs/classic/{algo}/{robot}/{run_name}/
  best_model/
    best_model.zip
    vec_normalize.pkl


## 5. 中断与继续训练

### 5.1 中断训练

- 第一次 `Ctrl+C`：保存 `interrupted` 并退出
- 第二次 `Ctrl+C`：立即强制退出

### 5.2 继续训练（默认从 interrupted）


In [ ]:
python classic/train.py \
  --algo sac \
  --robot ur5_cxy \
  --run-name exp_sac \
  --resume \
  --timesteps 300000 \
  --device cuda \
  --buffer-size 1000000 \
  --skip-replay-buffer


### 5.3 指定恢复路径继续训练


In [ ]:
python classic/train.py \
  --algo sac \
  --run-name exp_sac_resume \
  --resume \
  --robot ur5_cxy \
  --resume-model-path models/classic/sac/ur5_cxy/exp_sac/interrupted/model \
  --resume-normalize-path models/classic/sac/ur5_cxy/exp_sac/interrupted/vec_normalize.pkl \
  --resume-replay-path models/classic/sac/ur5_cxy/exp_sac/interrupted/replay_buffer.pkl \
  --timesteps 300000 \
  --device cuda


## 6. 测试训练结果

### 6.1 用训练脚本测试


In [ ]:
python classic/train.py --test --algo sac --robot ur5_cxy --run-name exp_sac --episodes 5 --render-mode human --device cuda


### 6.2 用独立脚本测试（按 run-name 自动解析）


In [ ]:
python classic/test.py \
  --algo sac \
  --robot ur5_cxy \
  --run-name exp_sac \
  --episodes 3 \
  --render-mode human


### 6.3 用独立脚本测试（指定路径）


In [ ]:
python classic/test.py \
  --algo sac \
  --robot ur5_cxy \
  --model-path models/classic/sac/ur5_cxy/exp_sac/final/model.zip \
  --norm-path models/classic/sac/ur5_cxy/exp_sac/final/vec_normalize.pkl \
  --episodes 3 \
  --render-mode human


## 7. 关键文件

- `env.py`：环境、奖励、渲染
- `train.py`：训练/测试主入口
- `classic/test.py`：推荐测试入口（支持 `--run-name` 自动找模型）
- `test_model.py`：根目录兼容测试入口
- `assets/robotiq_cxy/`：UR5 资源
- `assets/zero_arm/`：zero 与 zero+Robotiq 资源
- `PROJECT_USAGE_MTD3_STYLE.md`：完整使用手册（同风格）
